In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from utils import twh_to_ej_str, build_const_value_xml, build_const_techs_xml, write_text, twh_to_ej, xy, gw_to_twh
from pathlib import Path

* Data Source:
    * the 11th Basic Plan for Supply and Demand of Power (BPESD, `../resources/BPESD-11-20250313`)
    * 2023 Electricity Statistics of Korea (ES, `../resources/ES-93-KEPCO`)

* Implemented Input Files
    * `/input/policy/korea-2035/power/coal_ammonia_blend_const_value_cp.xml`
    * `/input/policy/korea-2035/power/coal_ammonia_blend_const_techs.xml`
    * `/input/policy/korea-2035/power/coal_CCS_const_value_ep.xml`
    * `/input/policy/korea-2035/power/coal_CCS_const_techs.xml`
    * `/input/policy/korea-2035/power/coal_const_value_cp.xml`
    * `/input/policy/korea-2035/power/coal_const_value_ep.xml`
    * `/input/policy/korea-2035/power/coal_const_techs.xml`
    * `/input/policy/korea-2035/power/coal_turnoff.xml`
    * `/input/policy/korea-2035/power/coal_shutdown_cp.xml`
    * `/input/policy/korea-2035/power/coal_shutdown_ep.xml`

# Coal

Coal-fired power generation (TWh) is reported in the BPESD for 2023, 2030, and 2035. Generation for 2025 is derived from observed data covering the period from December 2024 to November 2025, based on Monthly Energy Statistics (MES).

From 2030 onward, the BPESD’s coal generation figures are interpreted to include the coal input associated with ammonia co-firing. Because ammonia-based generation is classified as carbon-free in the BPESD, the coal input embedded in ammonia co-firing is assumed to be equivalent to four times the reported ammonia generation. Accordingly, the estimated coal contribution attributable to ammonia co-firing is subtracted from the reported coal generation, and the resulting adjusted values are implemented as the coal generation ceiling.

In [2]:
dictCapTWh = {2020: 198.1, 2023: 184.9, 2025: 168.4, 2030: 110.5, 2035: 88.9}

In [3]:
dictCapTWhEp = {2020: 198.1, 2023: 184.9, 2025: 168.4}
for year in range(2030, 2040, 5):
    dictCapTWhEp[year] = max(dictCapTWhEp[2025] * (1 - (year - 2025) / (2035 - 2025)), 0)
dictCapTWhEp

{2020: 198.1, 2023: 184.9, 2025: 168.4, 2030: 84.2, 2035: 0.0}

In [6]:
years_cap, values_cap = xy(dictCapTWh)
years_alt, values_alt = xy(dictCapTWhEp)

fig = go.Figure()

for name, x, y, dash in [
    ("Current Policies", years_cap, values_cap, None),
    ("Enhanced Ambition", years_alt, values_alt, "dash"),
]:
    fig.add_trace(go.Scatter(
        x=list(x), y=list(y),
        mode='lines+markers',
        name=name,
        line=(dict(dash=dash) if dash else None)
    ))

# Build annotations without repeating blocks
target_years = [2020, 2025, 2030, 2035]
annotations = []
for d in (dictCapTWh, dictCapTWhEp):
    for yr in target_years:
        val = d.get(yr)
        if val is not None:
            annotations.append(go.layout.Annotation(
                x=yr, y=val,
                xanchor='center', yanchor='bottom',
                text=f"{val:.1f} TWh",
                showarrow=True, arrowhead=1, ax=0, ay=-20
            ))

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    width=800, height=600,
    annotations=annotations,
    # xaxis=dict(title='Year', title_font=dict(size=18), tickfont=dict(size=15)),
    yaxis=dict(title='TWh',  title_font=dict(size=18), tickfont=dict(size=15)),
)
import plotly.io as pio
pio.write_image(fig, "../figure/coal.jpg", width=800, height=600, scale=3)
fig.show()

In [6]:
dictCapTWhAmmonia = {2020: 0, 2023: 0, 2025: 0, 2030: 15.5*(1-0.469), 2035: 32.8*(1-0.469)}

In [7]:
years = [2020, 2025, 2030, 2035]
values_cp = {y: twh_to_ej_str(dictCapTWh[y] - (dictCapTWhAmmonia[y] * 4)) for y in years}
values_ep = {y: twh_to_ej_str(dictCapTWhEp[y]) for y in years}
policy_name = "Coal-Ceiling"
policy_type = "tax"
subsector_name = 'coal'
tech_names = ['coal (IGCC)', 'coal (conv pul)']

In [8]:
xml_value_cp = build_const_value_xml(
    values_by_year=values_cp,
    policy_name=policy_name,
    policy_type=policy_type,
)

xml_value_ep = build_const_value_xml(
    values_by_year=values_ep,
    policy_name=policy_name,
    policy_type=policy_type,
)

xml_techs = build_const_techs_xml(
    years=[2020, 2025, 2030, 2035],
    sector_name="electricity",
    subsector_name=subsector_name,
    policy_name=policy_name,
    tech_names=tech_names,
    policy_type=policy_type,
)

In [9]:
value_path_cp = f"../../input/policy/korea-2035/power/coal_const_value_cp.xml"
value_path_ep = f"../../input/policy/korea-2035/power/coal_const_value_ep.xml"
techs_path = f"../../input/policy/korea-2035/power/coal_const_techs.xml"

write_text(value_path_cp, xml_value_cp)
write_text(value_path_ep, xml_value_ep)
write_text(techs_path, xml_techs)

print("Wrote:", Path(value_path_cp).expanduser())
print("Wrote:", Path(value_path_ep).expanduser())
print("Wrote:", Path(techs_path).expanduser())

Wrote: ../../input/policy/korea-2035/power/coal_const_value_cp.xml
Wrote: ../../input/policy/korea-2035/power/coal_const_value_ep.xml
Wrote: ../../input/policy/korea-2035/power/coal_const_techs.xml
